<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png"  width='15%'>  


<h1> Demo: BERT Fine Tuning</h1>
Developed by WeCloudData
<br></br>




## BERT Finetune

Here's a recent sample of the BERT Finetune using Huggingface models

This is the original guide and how you would implement it today: https://medium.com/@heyamit10/fine-tuning-bert-for-classification-a-practical-guide-b8c1c56f252c


In [1]:
!pip install transformers datasets==2.14.5 torch scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.6/519.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 15.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2023.6.0 which is incompatible.


In [2]:
#Check the availability of CUDA (GPU)

import torch
print(torch.cuda.is_available())

True


## Data Preprocessing

Not all data is created equally and what is important is to be able to **identify imblanaces in classes.** We want to avoid anything that may skew our dataset overall and cause problems at time of inference.


In [3]:
#Start with an old familiar friend, our IMDB datasets

from datasets import load_dataset

# Load the IMDB dataset
dataset = load_dataset("imdb")

# Peek at the dataset structure
print(dataset)

Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


A few things worth noting for practical purposes:
- Tokenization is the majority of the work since it is the vocabulary and how the defined keys and words are used in a sequence or problem
- Over-truncating sequences can hurt the performance of a model and being generous with padding can help performance. Of course, it's about balancing cost, accuracy and speed.
- Class imbalance can cause models to lean too much to one or another side. Hence why evaluating the balance of the classes is important. If 80% of reviews are positive vs. 20% negative reviews, you'll receive a lot of positive results instead of a clear assessment based on the actual context.

In [4]:
from transformers import AutoTokenizer
#answerdotai/ModernBERT-base
model_selected = "bert-base-uncased"
# Initialize the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_selected)

# Tokenize a sample text
sample_text = "I absolutely loved this movie! Highly recommend it."

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Apply the tokenizer to the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Inspect tokenized samples
print(tokenized_datasets["train"][0])


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [5]:
#This is a simple way to evaluate the class imbalances present in the dataset
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = tokenized_datasets["train"]["label"]
class_weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)
print(class_weights)

[1. 1.]


## Assessing Models

For the case of different models, it's important to see what the actual architecture is in case you need to adjust parts in training or fine tuning.  

Here are a few components to look out for:

- `architecture` - understand the fundamental design
- `num_labels` - in the definition of the model, it's important to assess the data at least a little bit to know the classes and values available.
- `..._dropout` - the different rates of dropout at the different stages / layers.



In [6]:
from transformers import AutoModelForSequenceClassification

# Initialize a BERT model for binary classification
model_name = "answerdotai/ModernBERT-base"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(model.config)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ModernBertConfig {
  "architectures": [
    "ModernBertForMaskedLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 50281,
  "classifier_activation": "gelu",
  "classifier_bias": false,
  "classifier_dropout": 0.0,
  "classifier_pooling": "mean",
  "cls_token_id": 50281,
  "decoder_bias": true,
  "deterministic_flash_attn": false,
  "dtype": "float32",
  "embedding_dropout": 0.0,
  "eos_token_id": 50282,
  "global_attn_every_n_layers": 3,
  "global_rope_theta": 160000.0,
  "gradient_checkpointing": false,
  "hidden_activation": "gelu",
  "hidden_size": 768,
  "initializer_cutoff_factor": 2.0,
  "initializer_range": 0.02,
  "intermediate_size": 1152,
  "layer_norm_eps": 1e-05,
  "local_attention": 128,
  "local_rope_theta": 10000.0,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "mlp_dropout": 0.0,
  "model_type": "modernbert",
  "norm_bias": false,
  "norm_eps": 1e-05,
  "num_attention_heads": 12,
  "num_hidden_layers": 22,
  "pad_token_id": 502

In [7]:
# # Example: Mapping string labels to integers
# label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
# dataset = dataset.map(lambda x: {"label": label_mapping[x["text_label"]]})

# # Verify the mapping
# print(dataset["train"][0])

### Adapting Layers

So, one of the methods of fine tuning, especially in definition of tasks, is to freeze layers for part of the architecture.

> *When do I "freeze" layers?*

- For smaller datasets (<10k samples), freeze the lower layers to avoid model performance degradation.
- For domain-specific tasks (document creation, tone / style adaptation), leave all layers flexible.

In [8]:
# # Freeze all layers except the classifier
# for param in model.bert.parameters():
#     param.requires_grad = False

# # Keep only the classification head trainable
# for param in model.classifier.parameters():
#     param.requires_grad = True

# print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

### Let's Complicate Things (For Learning Sake)

So, let's say a blanket application of layer freezing isn't going to cut it for us. We want more control over the actual model and the corresponding architecture.

This is where creating a new model class inheriting the majority of the model and adapting components is helpful

In [9]:
import torch.nn as nn

class CustomBERTModel(nn.Module):
    def __init__(self, pretrained_model_name, num_labels):
        super(CustomBERTModel, self).__init__()
        self.bert = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name, num_labels=num_labels)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = self.dropout(output[1])  # Applying dropout
        logits = self.fc(pooled_output)  # Adding a fully connected layer
        return logits

# Initialize the custom model
custom_model = CustomBERTModel(model_selected, num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


> **NOTE**   
> This might surprise you: sometimes, small architectural tweaks like this can lead to noticeable improvements, especially for imbalanced datasets or nuanced classification tasks.

In [11]:
from transformers import TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",           # Directory for saving model checkpoints
    eval_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",     # Save at the end of each epoch to match eval_strategy
    learning_rate=5e-5,              # Start with a small learning rate
    per_device_train_batch_size=16,  # Batch size per GPU
    per_device_eval_batch_size=16,
    num_train_epochs=3,              # Number of epochs
    weight_decay=0.01,               # Regularization
    save_total_limit=2,              # Limit checkpoints to save space
    report_to = "none",
    load_best_model_at_end=True,     # Automatically load the best checkpoint
    logging_dir="./logs",            # Directory for logs
    logging_steps=100,               # Log every 100 steps
    fp16=True                        # Enable mixed precision for faster training
)

print(training_args)

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,

### Define Metrics

How do we know we are doing well? Naturally, we have to select the metrics that make sense to evaluate the final performance of the model.

So, instead of relying on the out of the box metrics, let's showcase a basic example on how to apply our own metrics with a custom function. And this could use one or multiple metrics.


In [12]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [13]:
from transformers import Trainer
from evaluate import load

# Load a metric (F1-score in this case)
metric = load("f1")

# Define a custom compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [14]:
#Add data collator for efficient training of the model, especially when padding is involved
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [15]:
trainer = Trainer(
    model=model,                        # Pre-trained BERT model
    args=training_args,                 # Training arguments
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,        # Efficient batching
    compute_metrics=compute_metrics     # Custom metric
)

# Start training
trainer.train()

/tmp/ipykernel_3365/2180504376.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None, 'pad_token_id': 0}.
W0813 17:09:23.840000 3365 torch/_inductor/utils.py:1731] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,F1
1,0.438400,0.440399,0.775514
2,0.292800,0.451685,0.783933
3,0.169500,0.610619,0.804570


TrainOutput(global_step=4689, training_loss=0.32917477230853837, metrics={'train_runtime': 1341.7687, 'train_samples_per_second': 55.896, 'train_steps_per_second': 3.495, 'total_flos': 6389209612800000.0, 'train_loss': 0.32917477230853837, 'epoch': 3.0})